In [2]:
import joblib
import pandas as pd
import numpy as np

# Carrega os arquivos de modelo 
modelo = joblib.load('modelo_copa_2026.pkl')
scaler = joblib.load('scaler_copa_2026.pkl')

print("Modelo e Scaler carregados com sucesso! ")

Modelo e Scaler carregados com sucesso! 


In [3]:
# Função de previsao dos jogos

def prever_confronto(peso_torneio, delta_ranking, gols_man, gols_vis, conf_totais, vit_man, vit_vis, nome_man='Mandante', nome_vis='Visitante'):
    """"
    Recebe os atributos táticos das duas equipes, passa pelo Saclaer e devolve a probabilidade exata do placar (Vitoria Mandante, Empate, Vitoria Visitante)
     
    """
    # Deixar os dados no mesmo formato do treino
    dados_confronto = pd.DataFrame([{
        'PESO_COMPETICAO': peso_torneio,
        'DELTA_RANKING_PONTOS': delta_ranking,
        'MED_GOLS_MARCADOS_MANDANTE': gols_man,
        'MED_GOLS_MARCADOS_VISITANTE': gols_vis,
        'HISTORICO_CONFRONTOS_TOTAIS': conf_totais,
        'HISTORICO_VITORIAS_MANDANTE': vit_man,
        'HISTORICO_VITORIAS_VISITANTE': vit_vis       
    }])
    
    dados_scaled = scaler.transform(dados_confronto)
    
    # probabilidade para cada classe
    probs = modelo.predict_proba(dados_scaled)[0]
    
    print(f"🔮 Previsão: {nome_man} vs {nome_vis}")
    print(f"   -> Probabilidade de Vitória do {nome_man}: {round(probs[2] * 100, 2)}%")
    print(f"   -> Probabilidade de Empate: {round(probs[1] * 100, 2)}%")
    print(f"   -> Probabilidade de Vitória do {nome_vis}: {round(probs[0] * 100, 2)}%")
    print("-" * 50)
    
    return probs

In [4]:
# Executa o teste do motor preditor
probabilidades = prever_confronto(
    peso_torneio=3,     # Copa do Mundo
    delta_ranking=150.0, # Brasil superior no ranking
    gols_man=2.1,       # Média de gols do Brasil nos últimos 24 meses
    gols_vis=1.4,       # Média de gols da Itália nos últimos 24 meses
    conf_totais=5,      # Jogos históricos entre eles
    vit_man=3,          # Vitórias históricas do Brasil
    vit_vis=1,          # Vitórias históricas da Itália
    nome_man="Brasil",
    nome_vis="Itália"
)

🔮 Previsão: Brasil vs Itália
   -> Probabilidade de Vitória do Brasil: 64.85%
   -> Probabilidade de Empate: 17.46%
   -> Probabilidade de Vitória do Itália: 17.69%
--------------------------------------------------


In [5]:
import sqlalchemy

SERVER = 'FELIPE-PC\\SQLEXPRESS'
DATABASE = 'DB_COPA_2026'
connection_url = f"mssql+pyodbc://@{SERVER}/{DATABASE}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = sqlalchemy.create_engine(connection_url)

In [6]:
query_perfis = """
WITH CTE_Gols_Recentes AS (
    -- Consolida a média de gols dos últimos 24 meses
    SELECT 
        ID_SELECAO,
        AVG(GOLS_MARCADOS) AS MED_GOLS_RECENTE
    FROM (
        SELECT ID_SELECAO_MANDANTE AS ID_SELECAO, GOLS_MANDANTE AS GOLS_MARCADOS, DATA_PARTIDA FROM fato_partidas
        UNION ALL
        SELECT ID_SELECAO_VISITANTE AS ID_SELECAO, GOLS_VISITANTE AS GOLS_MARCADOS, DATA_PARTIDA FROM fato_partidas
    ) AS todas_partidas
    WHERE DATA_PARTIDA >= DATEADD(MONTH, -24, (SELECT MAX(DATA_PARTIDA) FROM fato_partidas))
    GROUP BY ID_SELECAO
),
CTE_Ultimo_Ranking AS (
    -- Mapeado com as colunas reais: 'team', '[total.points]' e 'date'
    SELECT 
        team,
        [total.points],
        ROW_NUMBER() OVER (PARTITION BY team ORDER BY date DESC) AS RN
    FROM stg_ranking_fifa
)
SELECT 
    s.ID_SELECAO,
    s.NOME_SELECAO,
    ISNULL(g.MED_GOLS_RECENTE, 0) AS MED_GOLS,
    ISNULL(r.[total.points], 1000) AS PONTOS_FIFA
FROM dim_selecoes s
LEFT JOIN CTE_Gols_Recentes g ON s.ID_SELECAO = g.ID_SELECAO
LEFT JOIN CTE_Ultimo_Ranking r ON s.NOME_SELECAO = r.team AND r.RN = 1;
"""

# Executa a carga com o mapeamento idêntico ao banco
df_perfis_selecoes = pd.read_sql(query_perfis, engine)
df_perfis_selecoes.set_index('NOME_SELECAO', inplace=True)

print(f"✅ Tabela de Consulta Automatizada criada! {df_perfis_selecoes.shape[0]} seleções tagueadas.")
df_perfis_selecoes.sample(10)

✅ Tabela de Consulta Automatizada criada! 364 seleções tagueadas.


,ID_SELECAO,MED_GOLS,PONTOS_FIFA
NOME_SELECAO,,,
Tahiti,313,1,1016.83
Åland Islands,3,0,1000.00
Romani people,252,0,1000.00
Croatia,80,2,1701.31
Italy,155,1,1714.29
North Korea,221,1,1000.00
Republic of Ireland,248,1,1403.84
Occitania,228,0,1000.00
Latvia,174,0,1095.98


In [7]:

print(pd.read_sql("SELECT TOP 1 * FROM stg_ranking_fifa", engine).columns.tolist())

['date', 'semester', 'rank', 'team', 'acronym', 'total.points', 'previous.points', 'diff.points']


In [8]:
import json

# Os 12 grupos oficiais e reais da Copa do Mundo de 2026
dados_dos_grupos = {
    "Grupo A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "Grupo B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "Grupo C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "Grupo D": ["USA", "Paraguay", "Australia", "Türkiye"],
    "Grupo E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "Grupo F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "Grupo G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "Grupo H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "Grupo I": ["France", "Senegal", "Iraq", "Norway"],
    "Grupo J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "Grupo K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "Grupo L": ["England", "Croatia", "Ghana", "Panama"]
}

# Salva os dados em um arquivo JSON externo
with open('grupos_copa_2026.json', 'w', encoding='utf-8') as f:
    json.dump(dados_dos_grupos, f, ensure_ascii=False, indent=4)

print("💾 Arquivo de configuração 'grupos_copa_2026.json' gerado com sucesso!")

💾 Arquivo de configuração 'grupos_copa_2026.json' gerado com sucesso!


In [9]:
# O Python lê o arquivo externo de forma limpa e profissional
with open('grupos_copa_2026.json', 'r', encoding='utf-8') as f:
    grupos_copa = json.load(f)

# Exibe o chaveamento para garantir que funcionou
for grupo, times in grupos_copa.items():
    print(f"{grupo}: {times}")

Grupo A: ['Mexico', 'South Africa', 'South Korea', 'Czech Republic']
Grupo B: ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland']
Grupo C: ['Brazil', 'Morocco', 'Haiti', 'Scotland']
Grupo D: ['USA', 'Paraguay', 'Australia', 'Türkiye']
Grupo E: ['Germany', 'Curaçao', 'Ivory Coast', 'Ecuador']
Grupo F: ['Netherlands', 'Japan', 'Sweden', 'Tunisia']
Grupo G: ['Belgium', 'Egypt', 'Iran', 'New Zealand']
Grupo H: ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay']
Grupo I: ['France', 'Senegal', 'Iraq', 'Norway']
Grupo J: ['Argentina', 'Algeria', 'Austria', 'Jordan']
Grupo K: ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia']
Grupo L: ['England', 'Croatia', 'Ghana', 'Panama']


In [10]:
def buscar_historico_confronto(nome_man, nome_vis):
    """
    Conecta ao banco de dados e recupera o histórico real de confrontos diretos
    entre as duas seleções utilizando a view que criamos no SQL Server.
    """
    query = f"""
    SELECT 
        HISTORICO_CONFRONTOS_TOTAIS,
        HISTORICO_VITORIAS_MANDANTE,
        HISTORICO_VITORIAS_VISITANTE
    FROM vw_confronto_direto
    WHERE ID_SELECAO_MANDANTE = (SELECT ID_SELECAO FROM dim_selecoes WHERE NOME_SELECAO = '{nome_man}')
      AND ID_SELECAO_VISITANTE = (SELECT ID_SELECAO FROM dim_selecoes WHERE NOME_SELECAO = '{nome_vis}')
    """
    try:
        df_h2h = pd.read_sql(query, engine)
        if not df_h2h.empty:
            return df_h2h.iloc[0].to_dict()
    except:
        pass
    
    # Se as seleções nunca se enfrentaram na história, retorna o dicionário zerado
    return {'HISTORICO_CONFRONTOS_TOTAIS': 0, 'HISTORICO_VITORIAS_MANDANTE': 0, 'HISTORICO_VITORIAS_VISITANTE': 0}

In [58]:
import itertools

# ⚙️ PARÂMETRO MESTRE: Defina 'estocastico' para emoção/zebras ou 'deterministico' para a lógica pura da IA
MODO_SIMULACAO = 'estocastico' 

classificacao_geral = {}
print(f"🏃‍♂️ Inicializando a Fase de Grupos 2026 no modo: [{MODO_SIMULACAO.upper()}]...\n")

for nome_grupo, times in grupos_copa.items():
    # Inicializa a tabela do grupo zerada
    tabela = pd.DataFrame(index=times, columns=['P', 'J', 'V', 'E', 'D', 'GP', 'GC', 'SG'])
    tabela.fillna(0, inplace=True)
    
    # Todos contra todos dentro do grupo
    confrontos = list(itertools.combinations(times, 2))
    
    for time_man, time_vis in confrontos:
        # Coleta os dados da nossa Lookup Table e do Banco
        perfis = df_perfis_selecoes.loc[[time_man, time_vis]]
        med_gols_man = perfis.loc[time_man, 'MED_GOLS']
        med_gols_vis = perfis.loc[time_vis, 'MED_GOLS']
        delta_ranking = perfis.loc[time_man, 'PONTOS_FIFA'] - perfis.loc[time_vis, 'PONTOS_FIFA']
        
        h2h = buscar_historico_confronto(time_man, time_vis)
        
        # Prepara os dados para o modelo
        dados_confronto = pd.DataFrame([{
            'PESO_COMPETICAO': 3,
            'DELTA_RANKING_PONTOS': delta_ranking,
            'MED_GOLS_MARCADOS_MANDANTE': med_gols_man,
            'MED_GOLS_MARCADOS_VISITANTE': med_gols_vis,
            'HISTORICO_CONFRONTOS_TOTAIS': h2h['HISTORICO_CONFRONTOS_TOTAIS'],
            'HISTORICO_VITORIAS_MANDANTE': h2h['HISTORICO_VITORIAS_MANDANTE'],
            'HISTORICO_VITORIAS_VISITANTE': h2h['HISTORICO_VITORIAS_VISITANTE']
        }])
        
        # Predição de Probabilidades
        dados_scaled = scaler.transform(dados_confronto)
        probs = modelo.predict_proba(dados_scaled)[0] # [Classe 0: Vis, Classe 1: Empate, Classe 2: Man]
        
        # 🧠 A GRANDE VIRADA: Decisão do resultado baseado no modo escolhido
        if MODO_SIMULACAO == 'deterministico':
            # Abordagem Absoluta: O resultado de maior probabilidade sempre ganha (Sem sorteio)
            resultado_final = np.argmax(probs) 
            
            # Gols calculados friamente pela média arredondada
            gols_man_sim = int(round(med_gols_man))
            gols_vis_sim = int(round(med_gols_vis))
        else:
            # Abordagem Estocástica: Sorteio ponderado (Gira a roleta da IA)
            resultado_final = np.random.choice([0, 1, 2], p=probs)
            
            # Gols calculados via distribuição estatística de Poisson (Gera variabilidade)
            gols_man_sim = int(np.random.poisson(med_gols_man))
            gols_vis_sim = int(np.random.poisson(med_gols_vis))
        
        # ⚖️ Ajuste Fino dos Gols: Garante que o placar reflete o resultado decretado
        if resultado_final == 2 and gols_man_sim <= gols_vis_sim: # Vitória Mandante
            gols_man_sim = gols_vis_sim + 1
        elif resultado_final == 0 and gols_vis_sim <= gols_man_sim: # Vitória Visitante
            gols_vis_sim = gols_man_sim + 1
        elif resultado_final == 1: # Empate
            gols_man_sim = gols_vis_sim = max(gols_man_sim, gols_vis_sim)
            
        # 📊 Atualização de estatísticas na tabela
        tabela.loc[time_man, 'J'] += 1
        tabela.loc[time_vis, 'J'] += 1
        tabela.loc[time_man, 'GP'] += gols_man_sim
        tabela.loc[time_man, 'GC'] += gols_vis_sim
        tabela.loc[time_vis, 'GP'] += gols_vis_sim
        tabela.loc[time_vis, 'GC'] += gols_man_sim
        
        if resultado_final == 2:
            tabela.loc[time_man, 'P'] += 3
            tabela.loc[time_man, 'V'] += 1
            tabela.loc[time_vis, 'D'] += 1
        elif resultado_final == 0:
            tabela.loc[time_vis, 'P'] += 3
            tabela.loc[time_vis, 'V'] += 1
            tabela.loc[time_man, 'D'] += 1
        else:
            tabela.loc[time_man, 'P'] += 1
            tabela.loc[time_vis, 'P'] += 1
            tabela.loc[time_man, 'E'] += 1
            tabela.loc[time_vis, 'E'] += 1
            
        tabela['SG'] = tabela['GP'] - tabela['GC']

    # Critérios de Desempate oficiais da FIFA
    classificacao_geral[nome_grupo] = tabela.sort_values(by=['P', 'V', 'SG', 'GP'], ascending=False)

print("🏆 Fase de Grupos processada com sucesso!")

🏃‍♂️ Inicializando a Fase de Grupos 2026 no modo: [ESTOCASTICO]...

🏆 Fase de Grupos processada com sucesso!


In [57]:
# Exibe a tabela final do Grupo C
for i, v in classificacao_geral.items():
    print(i)
    print(v)
    print('-' * 60)

Grupo A
                P  J  V  E  D GP GC  SG
Mexico          7  3  2  1  0  4  2   2
South Korea     4  3  1  1  1  4  4   0
Czech Republic  4  3  1  1  1  4  4   0
South Africa    1  3  0  1  2  2  4  -2
------------------------------------------------------------
Grupo B
                        P  J  V  E  D GP GC  SG
Bosnia and Herzegovina  9  3  3  0  0  6  3   3
Canada                  4  3  1  1  1  5  5   0
Qatar                   3  3  1  0  2  4  5  -1
Switzerland             1  3  0  1  2  4  6  -2
------------------------------------------------------------
Grupo C
          P  J  V  E  D GP GC  SG
Haiti     7  3  2  1  0  8  5   3
Brazil    7  3  2  1  0  7  5   2
Morocco   3  3  1  0  2  6  7  -1
Scotland  0  3  0  0  3  3  7  -4
------------------------------------------------------------
Grupo D
           P  J  V  E  D GP GC  SG
Australia  6  3  2  0  1  4  2   2
Paraguay   6  3  2  0  1  4  3   1
USA        6  3  2  0  1  3  3   0
Türkiye    0  3  0  0  3  0  3  -3
